In [1]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# URL del archivo en el bucket público de AWS
url = "https://hybridge-education-machine-learning-datasets.s3.us-east-1.amazonaws.com/Fraud.csv"
df = pd.read_csv(url)

In [3]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
len(df)

6362620

# ¡¡¡6,362,620 OBSERVACIONES!!!

# 1. Exploraciones de datos

In [5]:
# Revisar las dimensiones del conjunto de datos
print("Dimensiones del dataset:", df.shape)

# Revisar los nombres y tipos de columnas
print(df.info())

# Revisar si hay valores faltantes
print("\nValores faltantes:")
print(df.isnull().sum())

# Revisar la distribución de las clases
print("\nDistribución de las transacciones:")
print(df.iloc[:, -1].value_counts())

Dimensiones del dataset: (6362620, 11)
<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB
None

Valores faltantes:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Distribución de las transacciones:
isFlaggedFraud
0    6362604
1         16
Name: count, dtype: int64


# 2. Preparacion de los datos

In [6]:

# Eliminar columnas que identifican las cuentas
# y la columna que podría filtrar información sobre el fraude
df_model = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'])

# Convertir la columna type en variables numéricas
df_model = pd.get_dummies(df_model, columns=['type'], dtype=int)

# Separar variables predictoras y variable objetivo
X = df_model.drop(columns=['isFraud'])
y = df_model['isFraud']

# Revisar la distribución de las clases
print("Distribución de las clases:")
print(y.value_counts())

print("\nPorcentaje de fraude:")
print(f"{y.mean() * 100:.4f}%")

print("\nDimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)


Distribución de las clases:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Porcentaje de fraude:
0.1291%

Dimensiones de X: (6362620, 11)
Dimensiones de y: (6362620,)


# 3. Division de los datos y entrenar el modelo 

In [7]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Separar datos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Tamaño de entrenamiento:", X_train.shape)
print("Tamaño de prueba:", X_test.shape)

# Escalar las variables numéricas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Entrenar el modelo de regresión logística
modelo = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

modelo.fit(X_train_scaled, y_train)

# Realizar predicciones
y_pred = modelo.predict(X_test_scaled)

print("Modelo entrenado correctamente.")


Tamaño de entrenamiento: (5090096, 11)
Tamaño de prueba: (1272524, 11)
Modelo entrenado correctamente.


# 4. Evaluacion final del modelo

In [8]:

print("===== EVALUACIÓN FINAL DEL MODELO =====")

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nReporte de clasificación:")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Legítima', 'Fraudulenta'],
    zero_division=0
))


===== EVALUACIÓN FINAL DEL MODELO =====

Matriz de confusión:
[[1206261   64620]
 [     64    1579]]

Reporte de clasificación:
              precision    recall  f1-score   support

    Legítima       1.00      0.95      0.97   1270881
 Fraudulenta       0.02      0.96      0.05      1643

    accuracy                           0.95   1272524
   macro avg       0.51      0.96      0.51   1272524
weighted avg       1.00      0.95      0.97   1272524



#### Interpretacion
El modelo de regresión logística obtuvo una exactitud del 95 %. El recall para las transacciones fraudulentas fue del 96 %, lo que indica que logró identificar 1,579 de los 1,643 fraudes presentes en los datos de prueba.
Sin embargo, la precision para la clase fraudulenta fue del 2 %, debido a la gran cantidad de transacciones legítimas clasificadas incorrectamente como fraude.